<a href="https://colab.research.google.com/github/SaharMian/recsys-letterboxd/blob/main/nootbooks/nfc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movie recommendations on Letterboxd data, part 2: Neural Collaborative Filtering (NCF)

This notebook trains a **NeuMF** model (He et al., 2017) on the same data and the same train / validation / test split as part 1 (`Dataandeval.ipynb`). Run part 1 first, since this notebook reads the files it saves (`split_3000.parquet` and `results_val.csv`).

**This notebook covers:**
1. Loading the split and re-defining the evaluation
2. Turning ratings into "liked" examples and sampling negative examples
3. The NeuMF model
4. Training, and choosing the best epoch on validation
5. Adding the result to the comparison table
6. Final evaluation on the test set

**The idea:** SVD (part 1) learned to predict star ratings, and that turned out to favour obscure films. NCF is instead trained to answer "would this user like this film?" with a yes/no model, which is much closer to what we evaluate.

## Setup

Mount Google Drive, import the libraries and load the saved split from part 1. The integer ids `u` and `i` are created from the same categories as before, so they match part 1. Use a GPU runtime (T4).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

BASE = '/content/drive/MyDrive/recsys/'
SEED, LIKE, K = 42, 4.0, 10
device = 'cuda'
print("GPU available:", torch.cuda.is_available())

data = pd.read_parquet(BASE + 'split_3000.parquet')
data['u'] = data['user_name'].cat.codes
data['i'] = data['film_id'].cat.codes
n_users, n_films = data['u'].nunique(), data['i'].nunique()

train = data[data['split'] == 'train']
val   = data[data['split'] == 'val']
test  = data[data['split'] == 'test']

# Films each user has already rated in train (never recommended)
seen_train = train.groupby('user_name', observed=True)['film_id'].apply(set).to_dict()

print(n_users, n_films, data['split'].value_counts().to_dict())

Mounted at /content/drive
GPU available: True
3000 61555 {'train': 3552662, 'val': 444086, 'test': 442580}


## Evaluation and popularity baseline

The same evaluation as in part 1 (NDCG@10, Recall@10, Coverage), copied here so that this notebook can run on its own. The popularity baseline is repeated as a check: it must give exactly the same number as in part 1 (NDCG@10 0.1841).

In [2]:
def relevant_sets(df):
    liked = df[df['rating'] >= LIKE]
    return liked.groupby('user_name', observed=True)['film_id'].apply(set).to_dict()

def evaluate(recommend_fn, eval_df, seen):
    relevant = relevant_sets(eval_df)
    ndcgs, recalls, recommended = [], [], set()
    for user, rel in relevant.items():
        recs = recommend_fn(user, seen.get(user, set()), K)
        recommended.update(recs)
        hits = [1 if f in rel else 0 for f in recs]
        dcg  = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(rel), K)))
        ndcgs.append(dcg / idcg)
        recalls.append(sum(hits) / len(rel))
    return {'NDCG@10': np.mean(ndcgs), 'Recall@10': np.mean(recalls),
            'Coverage': len(recommended) / data['film_id'].nunique()}

# Popularity baseline: films ranked by number of likes in train
pop_rank = train[train['rating'] >= LIKE]['film_id'].value_counts().index.tolist()

def recommend_pop(user, seen, k):
    out = []
    for f in pop_rank:
        if f not in seen:
            out.append(f)
            if len(out) == k:
                break
    return out

print(evaluate(recommend_pop, val, seen_train))

{'NDCG@10': np.float64(0.18405938695253699), 'Recall@10': np.float64(0.03855764705308532), 'Coverage': 0.001364633254812769}


## Positive and negative examples

NCF does not predict star ratings. It answers a yes/no question: "would this user like this film?". We need examples of both answers:

- **Positive examples:** every train rating of 4.0 or higher (38% of train, 1.3 million rows).
- **Negative examples:** for every positive example we draw 4 random films that the user has **not rated at all** in train. This is called negative sampling. A random unrated film is not certain to be disliked, but most films are never seen by a given user, so it is a reasonable approximation.

`seen_mask` is a table of all (user, film) pairs with a rating in train, so we never draw a film the user has rated as a negative. The check at the end shows why this matters: the positives have a median of 462 ratings per film while the random negatives have only 11. The model will learn that popular films are more likely to be liked.

In [3]:
pos = train[train['rating'] >= LIKE]
pos_u = torch.tensor(pos['u'].values, dtype=torch.long, device=device)
pos_i = torch.tensor(pos['i'].values, dtype=torch.long, device=device)
print(f"Positive examples: {len(pos_u):,} ({len(pos_u) / len(train):.0%} of train)")

# Which (user, film) pairs have a rating in train, no matter how high
seen_mask = torch.zeros(n_users, n_films, dtype=torch.bool, device=device)
seen_mask[torch.tensor(train['u'].values, dtype=torch.long, device=device),
          torch.tensor(train['i'].values, dtype=torch.long, device=device)] = True

def sample_negatives(u, n_neg=4):
    """For each user id in u, draw n_neg random films the user has not rated in train"""
    u = u.repeat_interleave(n_neg)
    neg = torch.randint(0, n_films, u.shape, device=device)
    bad = seen_mask[u, neg]
    while bad.any():                      # redraw the ones that were already rated
        neg[bad] = torch.randint(0, n_films, (int(bad.sum()),), device=device)
        bad = seen_mask[u, neg]
    return u, neg

# Quick test on 1 million positives
torch.manual_seed(SEED)
u_neg, i_neg = sample_negatives(pos_u[:1_000_000])
print(f"Negative examples: {len(u_neg):,}")

film_count = torch.bincount(torch.tensor(train['i'].values, device=device), minlength=n_films)
print("Median ratings per film:  positives",
      film_count[pos_i].float().median().item(),
      "| negatives", film_count[i_neg].float().median().item())

Positive examples: 1,333,735 (38% of train)
Negative examples: 4,000,000
Median ratings per film:  positives 462.0 | negatives 11.0


## The NeuMF model and the recommendation function (untrained)

NeuMF (He et al., 2017) combines two ways of scoring a (user, film) pair, each with its own embeddings:

- **GMF:** the element-wise product of the user and film vectors. This is a generalised version of what SVD does.
- **MLP:** the user and film vectors are concatenated and sent through a small neural network (64 → 32 → 16 units, with ReLU). It can learn more complex combinations than a dot product.

The outputs of the two parts are joined and a final linear layer gives one number, the score for "will like". Higher means more likely.

`recommend_neumf` scores all 61,555 films for each user (in chunks of 50 users), removes films that are excluded (already seen) and returns the top 10.

Sanity check: an **untrained** model should be close to random. The result below (NDCG@10 about 0.001) confirms that the evaluation works and that the later numbers come from training.

In [4]:
class NeuMF(nn.Module):
    def __init__(self, n_users, n_films, d_gmf=32, d_mlp=32, layers=(64, 32, 16)):
        super().__init__()
        # separate embeddings for the GMF part and the MLP part
        self.Pg, self.Qg = nn.Embedding(n_users, d_gmf), nn.Embedding(n_films, d_gmf)
        self.Pm, self.Qm = nn.Embedding(n_users, d_mlp), nn.Embedding(n_films, d_mlp)
        mlp, in_dim = [], 2 * d_mlp
        for h in layers:
            mlp += [nn.Linear(in_dim, h), nn.ReLU()]
            in_dim = h
        self.mlp = nn.Sequential(*mlp)
        self.out = nn.Linear(d_gmf + layers[-1], 1)
        for emb in (self.Pg, self.Qg, self.Pm, self.Qm):
            nn.init.normal_(emb.weight, std=0.01)

    def forward(self, u, i):
        g = self.Pg(u) * self.Qg(i)                                   # GMF
        m = self.mlp(torch.cat([self.Pm(u), self.Qm(i)], dim=1))      # MLP
        return self.out(torch.cat([g, m], dim=1)).squeeze(1)

def recommend_neumf(model, exclude=('train',), k=K, chunk=50):
    """Top-k films per user, excluding the films the user has already seen"""
    ex = data[data['split'].isin(exclude)]
    ex_u = torch.tensor(ex['u'].values, dtype=torch.long, device=device)
    ex_i = torch.tensor(ex['i'].values, dtype=torch.long, device=device)
    users = data['user_name'].cat.categories
    films = data['film_id'].cat.categories
    all_films = torch.arange(n_films, device=device)
    model.eval()
    recs = {}
    with torch.no_grad():
        for start in range(0, n_users, chunk):
            end = min(start + chunk, n_users)
            u = torch.arange(start, end, device=device)
            scores = model(u.repeat_interleave(n_films), all_films.repeat(len(u))).view(len(u), n_films)
            sel = (ex_u >= start) & (ex_u < end)
            scores[ex_u[sel] - start, ex_i[sel]] = float('-inf')      # already seen
            top = scores.topk(k, dim=1).indices.cpu().numpy()
            for row, uid in enumerate(range(start, end)):
                recs[users[uid]] = [films[j] for j in top[row]]
    return recs

# Untrained model: should be close to random
torch.manual_seed(SEED)
model = NeuMF(n_users, n_films).to(device)
recs = recommend_neumf(model)
print(evaluate(lambda user, seen, k: recs[user], val, seen_train))

{'NDCG@10': np.float64(0.0007588090833367592), 'Recall@10': np.float64(0.00010598424943477677), 'Coverage': 0.009341239541873122}


## Training NeuMF

Each epoch we create a fresh set of negatives (4 per positive), mix positives and negatives in random order and train in batches. The loss is **binary cross-entropy** (`BCEWithLogitsLoss`): the model is punished for giving a low score to a liked film and a high score to a random one.

After every epoch we run the real evaluation (NDCG@10 on validation) and keep the weights of the best epoch. This is **early stopping**: the training loss keeps falling, but at some point the model starts to memorize the training data and the validation score gets worse. In our run the best epoch was 5 (val NDCG 0.2303), after which the score dropped while the coverage kept rising.

In [5]:
N_NEG, BATCH, LR, EPOCHS = 4, 16384, 0.002, 10

torch.manual_seed(SEED)
model = NeuMF(n_users, n_films).to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.BCEWithLogitsLoss()
best_ndcg, best_state, best_epoch = -1, None, 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    # new negatives every epoch
    u_neg, i_neg = sample_negatives(pos_u, N_NEG)
    u_all = torch.cat([pos_u, u_neg])
    i_all = torch.cat([pos_i, i_neg])
    y_all = torch.cat([torch.ones(len(pos_u)), torch.zeros(len(u_neg))]).to(device)
    perm = torch.randperm(len(y_all), device=device)

    total = 0.0
    for s in range(0, len(perm), BATCH):
        idx = perm[s:s + BATCH]
        loss = loss_fn(model(u_all[idx], i_all[idx]), y_all[idx])
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item() * len(idx)

    # evaluate on validation and keep the best epoch
    recs = recommend_neumf(model)
    res = evaluate(lambda user, seen, k: recs[user], val, seen_train)
    print(f"Epoch {epoch:>2}: loss {total / len(perm):.4f} | val NDCG {res['NDCG@10']:.4f}, "
          f"Recall {res['Recall@10']:.4f}, Coverage {res['Coverage']:.4f}")
    if res['NDCG@10'] > best_ndcg:
        best_ndcg, best_state, best_epoch = res['NDCG@10'], copy.deepcopy(model.state_dict()), epoch

model.load_state_dict(best_state)
torch.save(best_state, BASE + 'ncf_model.pt')
print(f"Best epoch: {best_epoch} (val NDCG {best_ndcg:.4f}), saved to Drive")

Epoch  1: loss 0.2911 | val NDCG 0.1941, Recall 0.0411, Coverage 0.0039
Epoch  2: loss 0.1985 | val NDCG 0.2015, Recall 0.0443, Coverage 0.0103
Epoch  3: loss 0.1770 | val NDCG 0.2185, Recall 0.0484, Coverage 0.0137
Epoch  4: loss 0.1585 | val NDCG 0.2261, Recall 0.0505, Coverage 0.0253
Epoch  5: loss 0.1409 | val NDCG 0.2303, Recall 0.0528, Coverage 0.0354
Epoch  6: loss 0.1273 | val NDCG 0.2297, Recall 0.0527, Coverage 0.0433
Epoch  7: loss 0.1188 | val NDCG 0.2174, Recall 0.0505, Coverage 0.0555
Epoch  8: loss 0.1127 | val NDCG 0.2219, Recall 0.0510, Coverage 0.0592
Epoch  9: loss 0.1083 | val NDCG 0.2206, Recall 0.0507, Coverage 0.0625
Epoch 10: loss 0.1050 | val NDCG 0.2143, Recall 0.0492, Coverage 0.0705
Best epoch: 5 (val NDCG 0.2303), saved to Drive


## Add NCF to the comparison table

We evaluate the best epoch on validation and add one row to `results_val.csv`, the table that part 1 created. The row has a fixed name, so running the notebook again replaces it instead of adding a duplicate.

Result: NCF reaches NDCG@10 **0.2303**, which beats the popularity baseline (0.1841) by about 25%, and also the best SVD variant (0.1929). Its coverage is 0.0354, far higher than popularity (0.0014) and SVD with a threshold (0.0010 to 0.0038). So NCF is both more accurate and more varied.

In [6]:
recs_ncf = recommend_neumf(model)
r = evaluate(lambda user, seen, k: recs_ncf[user], val, seen_train)

tbl = pd.read_csv(BASE + 'results_val.csv', index_col=0)
tbl.loc['NCF (NeuMF)'] = [round(r['NDCG@10'], 4), round(r['Recall@10'], 4), round(r['Coverage'], 4)]
tbl.to_csv(BASE + 'results_val.csv')
print(f"(best epoch: {best_epoch})")
tbl.round(4).

(best epoch: 5)


,NDCG@10,Recall@10,Coverage
Popularity (likes),0.184059,0.038558,0.001365
SVD (min 0 ratings),0.008587,0.002159,0.032345
SVD (min 1000 ratings),0.175731,0.036463,0.003818
SVD (min 1500 ratings),0.192934,0.040607,0.000958
NCF (NeuMF),0.230300,0.052800,0.035400


### What kind of films does NCF recommend?

Part 1 showed that SVD recommended films with a median of only 12 ratings. We make the same check for NCF. A median around 1200 ratings means it recommends well-known films, but not the very top of the popularity list (1630). That is a sign that it is personalising within the popular films instead of ignoring them.

In [7]:
tc = train['film_id'].value_counts().to_dict()   # number of train ratings per film

median_ncf = np.median([tc[f] for rec in recs_ncf.values() for f in rec])
print(f"Median train ratings per recommended film (NCF): {median_ncf:.0f}")

Median train ratings per recommended film (NCF): 1202


## Final evaluation on the test set

The test set has not been used for any choice so far (the epoch was chosen on validation). We now evaluate the chosen model **once** on test. As in part 1, films the user rated in train or validation count as seen and are excluded. The result is added to `results_test.csv`.

Result: NCF reaches NDCG@10 **0.3393** on test, about 32% better than the popularity baseline (0.2562) and about 43% better than the best SVD variant (0.2377). The improvement over popularity is similar to the one on validation (25%), so the choice of epoch on validation did not overfit. Remember that test numbers are higher than validation numbers for every model because of how the two protocols exclude films (see part 1), so compare models only within the same table.

In [8]:
TEST_PATH = BASE + 'results_test.csv'
COLS = ['NDCG@10', 'Recall@10', 'Coverage']

def add_test_row(name, res):
    """Add (or replace) a row in results_test.csv"""
    if os.path.exists(TEST_PATH):
        df = pd.read_csv(TEST_PATH, index_col=0)
    else:
        df = pd.DataFrame(columns=COLS)
    df.loc[name] = [res[c] for c in COLS]
    df.to_csv(TEST_PATH)
    return df

# Films each user has seen in train + validation (excluded at test time)
seen_tv = data[data['split'].isin(['train', 'val'])] \
    .groupby('user_name', observed=True)['film_id'].apply(set).to_dict()

In [9]:
# The model currently holds the weights of the best epoch (chosen on validation)
recs_test = recommend_neumf(model, exclude=('train', 'val'))
res = evaluate(lambda user, seen, k: recs_test[user], test, seen_tv)
df_test = add_test_row('NCF (NeuMF)', res)
df_test.round(4)

,NDCG@10,Recall@10,Coverage
Popularity (likes),0.2562,0.0493,0.0024
SVD (min 0 ratings),0.0106,0.0022,0.0324
SVD (min 1500 ratings),0.2377,0.0459,0.0010
NCF (NeuMF),0.3393,0.0685,0.0387
